## 01. 콘텐츠 기반 추천 시스템

콘텐츠 기반 추천은 아이템 자체의 정보를 이용해 비슷한 아이템을 추천하는 방식이다.

영화 추천을 예로 들면 사용자가 특정 영화를 좋아했을 때, 그 영화와 장르, 키워드, 줄거리 같은 정보가 비슷한 영화를 찾아 추천한다.

---

**왜 배우는가?**

추천 시스템을 처음 만들 때는 사용자 행동 데이터가 충분하지 않은 경우가 많다.  
하지만 영화의 장르, 상품의 카테고리, 여행지의 태그처럼 아이템 정보는 비교적 쉽게 구할 수 있다.

콘텐츠 기반 추천은 이런 아이템 정보를 숫자 벡터로 바꾼 뒤, 벡터가 비슷한 아이템을 찾아 추천한다.

---

**어디에 사용하는가?**

- 영화/음악/도서 추천: 장르, 키워드, 줄거리 기반 추천
- 쇼핑몰 추천: 상품명, 카테고리, 설명, 태그 기반 추천
- 뉴스 추천: 기사 제목, 본문, 주제 기반 추천
- 여행지 추천: 지역, 테마, 활동 유형 기반 추천

---

**이 노트북에서 사용하는 핵심 도구**

- `CountVectorizer`: 영화 장르나 키워드 문자열을 숫자 벡터로 바꾼다.
- `cosine_similarity`: 영화 벡터끼리 방향이 얼마나 비슷한지 계산한다.
- `Weighted Rating`: 평균 평점뿐 아니라 평가 수까지 반영해 추천 후보의 신뢰도를 보정한다.

콘텐츠 기반 추천에서는 별도의 학습 모델을 만들기보다, 아이템 feature를 벡터로 만들고 유사도가 높은 아이템을 찾아 정렬하는 흐름이 핵심이다.


## 02. 콘텐츠 기반 필터링 처리 흐름

https://www.kaggle.com/datasets/tmdb/tmdb-movie-metadata

콘텐츠 기반 필터링의 흐름은 다음과 같다.

1. 영화의 장르와 키워드 정보를 준비한다.
2. 문자열로 저장된 장르/키워드 구조를 파싱한다.
3. 장르 이름을 하나의 텍스트 feature로 만든다.
4. `CountVectorizer`로 텍스트를 숫자 벡터로 변환한다.
5. 코사인 유사도로 영화 간 유사도를 계산한다.
6. 기준 영화와 비슷한 후보를 찾고, 평점 또는 가중평점으로 정렬한다.

데이터셋 참고: TMDB 5000 Movie Dataset


## 03. 실습 환경 준비

데이터 처리, 텍스트 벡터화, 유사도 계산, 결과 확인에 필요한 라이브러리를 불러온다.


In [19]:
import numpy as np

import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
from pandas.core.array_algos import quantile


## 04. TMDB 영화 데이터셋 정보

이번 노트북은 `data/tmdb_5000_movies.csv` 파일을 사용한다.

원본 CSV 기준 데이터 구조는 다음과 같다.

- 데이터 수: 4,806행
- 컬럼 수: 20개
- 데이터 단위: 영화 1편이 1행
- 추천 방식: 영화의 장르/키워드 같은 아이템 속성을 이용해 비슷한 영화를 찾음

원본 컬럼은 다음과 같다.

| 컬럼명 | 의미 | 실습에서의 사용 |
|---|---|---|
| `budget` | 영화 제작 예산 | 직접 사용하지 않음 |
| `genres` | 영화 장르 목록 | 장르 기반 유사도 계산에 사용 |
| `homepage` | 영화 홈페이지 URL | 직접 사용하지 않음 |
| `id` | TMDB 영화 ID | 영화 식별용 |
| `keywords` | 영화 키워드 목록 | 추가 콘텐츠 feature로 확인 |
| `original_language` | 원어 코드 | 직접 사용하지 않음 |
| `original_title` | 원제 | 직접 사용하지 않음 |
| `overview` | 영화 설명 | 콘텐츠 feature로 확장 가능 |
| `popularity` | 인기도 점수 | 참고 정보 |
| `production_companies` | 제작사 정보 | 직접 사용하지 않음 |
| `production_countries` | 제작 국가 | 직접 사용하지 않음 |
| `release_date` | 개봉일 | 직접 사용하지 않음 |
| `revenue` | 수익 | 직접 사용하지 않음 |
| `runtime` | 상영 시간 | 직접 사용하지 않음 |
| `spoken_languages` | 사용 언어 | 직접 사용하지 않음 |
| `status` | 개봉 상태 | 직접 사용하지 않음 |
| `tagline` | 영화 홍보 문구 | 직접 사용하지 않음 |
| `title` | 영화 제목 | 추천 결과 표시와 기준 영화 검색에 사용 |
| `vote_average` | 평균 평점 | 추천 후보 정렬 기준으로 사용 |
| `vote_count` | 평가 수 | 가중평점 계산에 사용 |

노트북에서는 전체 컬럼 중 다음 컬럼만 선택해 사용한다.

```python
['id', 'title', 'genres', 'vote_average', 'vote_count', 'popularity', 'keywords', 'overview']
```

실습에서 중요한 컬럼은 `title`, `genres`, `keywords`, `vote_average`, `vote_count`이다.  
특히 `genres`와 `keywords`는 JSON 형태 문자열로 저장되어 있어 바로 유사도 계산에 사용할 수 없고, 먼저 장르명과 키워드명만 꺼내는 전처리가 필요하다.


## 05. 데이터 로드와 기본 확인

TMDB 영화 데이터에서 추천에 사용할 컬럼만 선택한다.

- `title`: 영화 제목
- `genres`: 장르 정보
- `keywords`: 키워드 정보
- `vote_average`: 평균 평점
- `vote_count`: 평가 수
- `popularity`: 인기도
- `overview`: 영화 설명

콘텐츠 기반 추천에서는 `genres`, `keywords`, `overview`처럼 아이템 자체를 설명하는 컬럼이 중요하다.


In [20]:
movie_df = pd.read_csv('data/tmdb_5000_movies.csv')
columns = ['id', 'title', 'genres', 'vote_average', 'vote_count', 'popularity', 'keywords', 'overview']
movie_df = movie_df[columns]
movie_df.head()


,id,title,genres,vote_average,vote_count,popularity,keywords,overview
0,19995,Avatar,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",7.2,11800,150.437577,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","In the 22nd century, a paraplegic Marine is di..."
1,285,Pirates of the Caribbean: At World's End,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",6.9,4500,139.082615,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","Captain Barbossa, long believed to be dead, ha..."
2,206647,Spectre,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",6.3,4466,107.376788,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",A cryptic message from Bond’s past sends him o...
3,49026,The Dark Knight Rises,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",7.6,9106,112.312950,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",Following the death of District Attorney Harve...
4,49529,John Carter,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",6.1,2124,43.926995,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...","John Carter is a war-weary, former military ca..."


## 06. 데이터 구조 확인

각 컬럼의 자료형과 결측치 여부를 확인한다.

장르와 키워드는 JSON처럼 보이는 문자열로 저장되어 있으므로, 바로 벡터화하기 전에 파싱이 필요하다.


In [21]:
movie_df.info()


<class 'pandas.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            4803 non-null   int64  
 1   title         4803 non-null   str    
 2   genres        4803 non-null   str    
 3   vote_average  4803 non-null   float64
 4   vote_count    4803 non-null   int64  
 5   popularity    4803 non-null   float64
 6   keywords      4803 non-null   str    
 7   overview      4800 non-null   str    
dtypes: float64(2), int64(2), str(4)
memory usage: 3.4 MB


## 07. 장르 문자열 구조 변환

`genres` 컬럼은 문자열처럼 보이지만 내부에는 리스트와 딕셔너리 구조가 들어 있다.

예를 들어 하나의 영화에 여러 장르가 들어 있고, 각 장르는 `id`, `name` 같은 정보를 가진다.  
추천에는 장르 이름만 필요하므로 먼저 문자열을 파이썬 객체로 바꾸고, 이후 `name` 값만 꺼낸다.


In [27]:
print(movie_df['genres'])

# literal_eval : 전달된 문자열 파이썬 코드를 실행하고 결과를 반환
from ast import literal_eval

movie_df['genres'] = movie_df['genres'].apply(
    lambda x: literal_eval(x) if isinstance(x, str) else x
)

print(movie_df['genres'].dtype)
print(type(movie_df['genres'].iloc[0]))

0       Action Adventure Fantasy Science Fiction
1                       Adventure Fantasy Action
2                         Action Adventure Crime
3                    Action Crime Drama Thriller
4               Action Adventure Science Fiction
                          ...                   
4798                       Action Crime Thriller
4799                              Comedy Romance
4800               Comedy Drama Romance TV Movie
4801                                            
4802                                 Documentary
Name: genres, Length: 4803, dtype: str


SyntaxError: invalid syntax (<unknown>, line 1)

## 08. 키워드 문자열 구조 변환

`keywords` 컬럼도 장르와 같은 방식으로 문자열 형태의 리스트/딕셔너리 구조를 가지고 있다.

키워드는 영화의 세부 주제나 특징을 담고 있으므로, 장르보다 더 세밀한 추천 feature로 확장할 수 있다.


In [23]:
from ast import literal_eval

movie_df['keywords'] = movie_df['keywords'].apply(
    lambda x: literal_eval(x) if isinstance(x, str) else x
)

print(type(movie_df['keywords'].iloc[0]))

<class 'list'>


## 09. 추천용 텍스트 feature 생성

장르와 키워드 리스트에서 `name` 값만 꺼내 공백으로 연결한다.

`CountVectorizer`는 문장을 토큰 단위로 나누어 숫자 벡터를 만들기 때문에, 여러 장르명을 하나의 문자열로 합쳐두면 벡터화에 사용할 수 있다.


In [29]:
# print(movie_df['genres'][0]

movie_df = pd.read_csv('data/tmdb_5000_movies.csv')

from ast import literal_eval

movie_df['genres'] = movie_df['genres'].apply(literal_eval)
movie_df['keywords'] = movie_df['keywords'].apply(literal_eval)

movie_df['genres'] = movie_df['genres'].apply(
    lambda x: ' '.join(g['name'] for g in x)
)

movie_df['keywords'] = movie_df['keywords'].apply(
    lambda x: ' '.join(g['name'] for g in x)
)

movie_df[['genres', 'keywords']].head()

,genres,keywords
0,Action Adventure Fantasy Science Fiction,culture clash future space war space colony so...
1,Adventure Fantasy Action,ocean drug abuse exotic island east india trad...
2,Action Adventure Crime,spy based on novel secret agent sequel mi6 bri...
3,Action Crime Drama Thriller,dc comics crime fighter terrorist secret ident...
4,Action Adventure Science Fiction,based on novel mars medallion space travel pri...


### 장르 기반 코사인 유사도 계산

영화의 장르 문자열을 숫자 벡터로 바꾼 뒤, 영화끼리 코사인 유사도를 계산한다.

여기서 코사인 유사도는 모델의 성능 평가 지표가 아니라 “두 영화가 장르 관점에서 얼마나 비슷한지”를 나타내는 유사도 점수이다. 값이 클수록 추천 후보로 볼 가능성이 높아진다.


## 10. 텍스트 벡터화

문자열은 그대로는 유사도 계산에 사용할 수 없다. 그래서 `CountVectorizer`를 이용해 장르 문자열을 숫자 벡터로 바꾼다.

예를 들어 `Action Adventure`라는 문자열은 `Action`과 `Adventure` 토큰이 있는 벡터로 바뀐다. 이후 영화 하나는 장르 토큰별 등장 여부나 등장 횟수를 가진 숫자 벡터로 표현된다.


In [30]:
from sklearn.feature_extraction.text import CountVectorizer

# CountVectorizer()
# - 학습한 문자열 데이터를 단어 또는 토큰의 등장 횟수 벡터로 변환한다
# - 영화 장르 / 키워드 / 줄거리 같은 문장형 텍스트를 숫자형 벡터로 변환할때 사용


# ngram_range=(1,2)
# - 단언 1개짜리 표현과 2개 연소 표현을 함께 사용한다는 의미
# ex) 'Science Fiction' 이 있으면
#     -> 'Science', 'Fiction' , 'Science Fiction' 함께 고려할 수 있게 함

cnt_vectorizer = CountVectorizer(ngram_range=(1, 2))

# 학습 + 변환(벡터화)
# 1. 전체 영화 장르 문자열에서 토큰 사전을 학습한다
# 2. 각 영화의 장르 문자열을 토큰 등장 횟수 벡터로 변환한다
# 3. 변환된 결과는 영화 수 * 토큰 수 형태의 spares matrix가 된다
# - spares matrix(희소 행렬) : 대부분의 원소가 0으로 채워진 행렬
movie_genre_mat = cnt_vectorizer.fit_transform(movie_df['genres'])

# 행(4803) - 영화 개수
# 열(276)  - 생성된 장르 토큰의 수
print(movie_genre_mat.shape)

# movie_genre_mat[:20. :20]

(4803, 276)


## 11. 벡터화 사전 확인

`CountVectorizer`가 만든 토큰 사전을 확인한다.

숫자 벡터의 각 컬럼이 어떤 장르 또는 장르 조합을 의미하는지 알기 위해 확인하는 단계이다.


In [31]:
# cnt_vectorizer.vocaulary_
# - CountVectorizer가 텍스트에서 찾은 토큰과 컬럼 번호를 매핑한 딕셔너리
# - 이 사전이 있어야지만 벡터의 각 컬럼이 어떤 단어를 뜻하는지 해석할 수 있다

cnt_vectorizer.vocabulary_

{'action': 0,
 'adventure': 16,
 'fantasy': 124,
 'science': 232,
 'fiction': 138,
 'action adventure': 1,
 'adventure fantasy': 24,
 'fantasy science': 135,
 'science fiction': 233,
 'fantasy action': 125,
 'crime': 64,
 'adventure crime': 20,
 'drama': 90,
 'thriller': 234,
 'action crime': 4,
 'crime drama': 68,
 'drama thriller': 106,
 'adventure science': 29,
 'animation': 33,
 'family': 109,
 'animation family': 38,
 'fantasy family': 130,
 'action science': 12,
 'adventure action': 17,
 'action thriller': 13,
 'thriller crime': 238,
 'western': 265,
 'adventure western': 32,
 'adventure family': 23,
 'family fantasy': 115,
 'fiction action': 139,
 'action fantasy': 7,
 'comedy': 44,
 'action comedy': 3,
 'comedy science': 59,
 'adventure drama': 22,
 'drama action': 91,
 'romance': 214,
 'drama romance': 104,
 'romance thriller': 228,
 'thriller action': 235,
 'fiction thriller': 150,
 'adventure thriller': 30,
 'fantasy adventure': 126,
 'family adventure': 111,
 'adventure com

## 12. 코사인 유사도 계산

코사인 유사도는 두 벡터의 방향이 얼마나 비슷한지 계산한다.

콘텐츠 기반 추천에서는 장르 벡터의 방향이 비슷한 영화들을 “콘텐츠가 비슷한 영화”로 해석한다. 이 값은 정답과 비교하는 평가 지표가 아니라, 추천 후보를 찾기 위한 정렬 기준이다.


In [33]:
from sklearn.metrics.pairwise import cosine_similarity

# cosine_similarity(A, B)
# - A, B 두 벡터의 방향 유사도를 계산
# - 만약 A, B가 같은 행렬인 경우
#   두 행렬의 모든 행과 열 사이의 유사도를 계산하는것
#   ex) 모든 영화(A)와 모든 영화(B) 사이의 유사도를 계산
genre_sim = cosine_similarity(movie_genre_mat, movie_genre_mat)

# 위 결과의 shape는 (영화 개수, 영화 개수)
# 행 i, 열 j == (i, j) == i 번째 영화 장르와 j번째 영화 장르의 유사도
print(genre_sim.shape) # (4803, 4803)

genre_sim[:10, :10]


(4803, 4803)


array([[1.        , 0.59628479, 0.4472136 , 0.12598816, 0.75592895,
        0.59628479, 0.        , 0.75592895, 0.4472136 , 0.74535599],
       [0.59628479, 1.        , 0.4       , 0.16903085, 0.3380617 ,
        0.8       , 0.        , 0.3380617 , 0.6       , 0.8       ],
       [0.4472136 , 0.4       , 1.        , 0.3380617 , 0.50709255,
        0.6       , 0.        , 0.50709255, 0.2       , 0.6       ],
       [0.12598816, 0.16903085, 0.3380617 , 1.        , 0.14285714,
        0.16903085, 0.        , 0.14285714, 0.        , 0.16903085],
       [0.75592895, 0.3380617 , 0.50709255, 0.14285714, 1.        ,
        0.50709255, 0.        , 1.        , 0.16903085, 0.50709255],
       [0.59628479, 0.8       , 0.6       , 0.16903085, 0.50709255,
        1.        , 0.        , 0.50709255, 0.4       , 0.8       ],
       [0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 1.        , 0.        , 0.25819889, 0.        ],
       [0.75592895, 0.3380617 , 0.5070925

## 13. 유사도 정렬 인덱스 생성

코사인 유사도 행렬에서 각 영화별로 유사도가 높은 영화 인덱스를 미리 정렬한다.

추천할 때마다 전체 유사도를 다시 정렬하지 않고, 정렬된 인덱스를 이용해 빠르게 후보를 가져올 수 있다.


In [37]:
# argsort(axis=-1)
# argsort() : 정렬했을 때 인덱스 순서 반환
# [10, 30, 20] 을 argsort() -> 0, 2, 1

# axis = -1 : 마지막 축 방향으로 연산을 수행하라
# == 현재는 열 방향

# argsort(axis=-1):각 행의 값을 작은 순서대로 정렬했을 때 인덱스 목록을 반환

# [:, ::-1] : 각 행의 정렬 결과를 뒤집어 유사도가 큰 영화부터 나열

# argsort

movie_idx_by_genre_sim_desc = genre_sim.argsort(axis=-1)[:, ::-1]

# 0,1,2 영화와 장르 유사도가 높은 순서대로 영화 인덱스 상위 10개 확인
for row in range(3):
    print(movie_idx_by_genre_sim_desc[row, :10])

[   0 3494  813  870   46   14 1296 1652  419  420]
[ 262    1  129  199  206  208   98 2343 2390  379]
[   2 1740 1542 1082  873 1615  969 1058 1073 1076]


## 14. 기준 영화의 유사 후보 확인

숫자 인덱스만 보면 추천 결과를 이해하기 어렵다.

기준 영화와 장르 유사도가 높은 후보를 실제 영화 제목, 장르, 평점과 함께 확인한다.


In [38]:
# 0번 영화(Avatar)와 장르 코사인 유사도가 가장 높은 10
movie_df.iloc[movie_idx_by_genre_sim_desc[0, :10]]

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,Action Adventure Fantasy Science Fiction,http://www.avatarmovie.com/,19995,culture clash future space war space colony so...,en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
3494,6000000,Action Adventure Fantasy Science Fiction,NaN,27549,based on novel time travel sequel psychotronic...,en,Beastmaster 2: Through the Portal of Time,"Mark Singer returns as Dar, the warrior who ca...",1.478505,"[{""name"": ""Les Films 21"", ""id"": 1113}, {""name""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",1991-06-08,869325,107.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,NaN,Beastmaster 2: Through the Portal of Time,4.6,17
813,55000000,Action Adventure Fantasy Science Fiction,NaN,1924,saving the world journalist dc comics crime fi...,en,Superman,Mild-mannered Clark Kent works as a reporter a...,48.507081,"[{""name"": ""Warner Bros."", ""id"": 6194}, {""name""...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",1978-12-13,300218018,143.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,You'll Believe a Man Can Fly!,Superman,6.9,1022
870,54000000,Action Adventure Fantasy Science Fiction,NaN,8536,saving the world dc comics sequel superhero ba...,en,Superman II,Three escaped criminals from the planet Krypto...,30.515175,"[{""name"": ""Warner Bros."", ""id"": 6194}, {""name""...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",1980-12-04,190458706,127.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Man of Steel meets his match!,Superman II,6.5,629
46,250000000,Action Adventure Fantasy Science Fiction,http://www.x-menmovies.com/,127585,1970s mutant time travel marvel comic based on...,en,X-Men: Days of Future Past,The ultimate X-Men ensemble fights a war for t...,118.078691,"[{""name"": ""Twentieth Century Fox Film Corporat...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2014-05-15,747862775,131.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"To save the future, they must alter the past",X-Men: Days of Future Past,7.5,6032
14,225000000,Action Adventure Fantasy Science Fiction,http://www.manofsteel.com/,49521,saving the world dc comics superhero based on ...,en,Man of Steel,A young boy learns that he has extraordinary p...,99.398009,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2013-06-12,662845518,143.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,You will believe that a man can fly.,Man of Steel,6.5,6359
1296,39000000,Comedy Action Adventure Fantasy Science Fiction,NaN,9531,saving the world dc comics super computer iden...,en,Superman III,"Aiming to defeat the Man of Steel, wealthy exe...",22.164202,"[{""name"": ""Warner Bros."", ""id"": 6194}, {""name""...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",1983-06-17,75850624,125.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,If the world's most powerful computer can cont...,Superman III,5.3,490
1652,100000000,Action Adventure Fantasy Science Fiction Thriller,NaN,14164,karate superhero revenge dragon duringcreditss...,en,Dragonball Evolution,The young warrior Son Goku sets out on a quest...,21.677732,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-04-01,0,85.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,The legend comes to life.,Dragonball Evolution,2.9

## 15. 추천 후보 정렬

유사도 계산이 끝나면 추천은 “정렬 문제”가 된다.

1. 기준 영화와 장르가 비슷한 후보를 찾는다.
2. 후보 중 기준 영화 자기 자신을 제외한다.
3. 후보를 평점 또는 가중평점 기준으로 다시 정렬한다.

초반에는 평균 평점(`vote_average`)을 사용하고, 뒤쪽에서는 평가 수까지 반영한 `weighted_vote_average`를 사용한다.


In [41]:
# title : 추천 기준이 되는 영화 제목
title = 'The Godfather'

# 영화 목록(movie_df)에서 title이 일치하는 인덱스 찾기
index = movie_df[movie_df['title'] == title].index[0]

# 기준 영화와 장르 유사도가 높은 영화 상위 20개의 인덱스 목록 조회
rcmd_movie_df = movie_df.iloc[movie_idx_by_genre_sim_desc[index, :20]]

rcmd_movie_df[['title', 'genres']]

# 후보 영화 중 편균 평점이 높은 영화 순서대로 정렬
rcmd_movie_df.sort_values(by='vote_average', ascending=False)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
1881,25000000,Drama Crime,NaN,278,prison corruption police brutality prison cell...,en,The Shawshank Redemption,Framed in the 1940s for the double murder of h...,136.747729,"[{""name"": ""Castle Rock Entertainment"", ""id"": 97}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",1994-09-23,28341469,142.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,Fear can hold you prisoner. Hope can set you f...,The Shawshank Redemption,8.5,8205
3337,6000000,Drama Crime,http://www.thegodfather.com/,238,italy love at first sight loss of father patri...,en,The Godfather,"Spanning the years 1945 to 1955, a chronicle o...",143.659698,"[{""name"": ""Paramount Pictures"", ""id"": 4}, {""na...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",1972-03-14,245066411,175.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,An offer you can't refuse.,The Godfather,8.4,5893
2731,13000000,Drama Crime,NaN,240,italo-american cuba vororte melancholy praise ...,en,The Godfather: Part II,In the continuing saga of the Corleone crime f...,105.792936,"[{""name"": ""Paramount Pictures"", ""id"": 4}, {""na...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",1974-12-20,47542841,200.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,"I don't feel I have to wipe everybody out, Tom...",The Godfather: Part II,8.3,3338
1847,25000000,Drama Crime,http://www.warnerbros.com/goodfellas,769,prison based on novel florida 1970s mass murde...,en,GoodFellas,"The true story of Henry Hill, a half-Irish, ha...",63.654244,"[{""name"": ""Winkler Films"", ""id"": 8880}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",1990-09-12,46836394,145.0,"[{""iso_639_1"": ""it"", ""name"": ""Italiano""}, {""is...",Released,Three Decades of Life in the Mafia.,GoodFellas,8.2,3128
1663,30000000,Drama Crime,NaN,311,life and death corruption street gang rape sad...,en,Once Upon a Time in America,A former Prohibition-era Jewish gangster retur...,49.336397,"[{""name"": ""Warner Bros."", ""id"": 6194}, {""name""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",1984-02-16,0,229.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,"Crime, passion and lust for power - Sergio Leo...",Once Upon a Time in America,8.2,1069
3866,3300000,Drama Crime,http://cidadededeus.globo.com/,598,male nudity street gang brazilian photographer...,pt,Cidade de Deus,Cidade de Deus is a shantytown that started du...,44.356711,"[{""name"": ""O2 Filmes"", ""id"": 345}, {""name"": ""V...","[{""iso_3166_1"": ""BR"", ""name"": ""Brazil""}, {""iso...",2002-02-05,30641770,130.0,"[{""iso_639_1"": ""pt"", ""name"": ""Portugu\u00eas""}]",Released,"If you run you're dead... if you stay, you're ...",City of God,8.1,1814
892,52000000,Drama Crime,NaN,524,poker drug abuse 1970s overdose illegal prosti...,en,Casino,The life of the gambling paradise – Las Vegas ...,40.066880,"[{""name"": ""Universal Pictures"", ""id"": 33}, {""n...","[{""iso_3166_1"": ""FR"", ""name"": ""France""}, {""iso...",1995-11-22,116112375,178.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,No one stays at the top forever.,Casino,7.8,1307
281,100000000,Drama Crime,http://www.americangangster.net/,4982,underdog black people drug traffic drug smuggl...,en,American Gangster,Following the death of his employer and mentor...,42.361215,"[{""name"": ""Imagine Entertainment"", ""id"": 23}, ...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2007-11-02,266465037,157.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,There are two sides to the American dream,American Gangster,7.4,1502
4041,2380000,Drama Crime,http://www.thisisenglandmovie.co.uk/,11798,holiday skinhead england vandalism independent...,

## 16. 장르 유사도 기반 추천 함수 생성

영화 제목을 입력하면 장르가 비슷한 영화를 찾아 평균 평점 기준으로 정렬하는 함수를 만든다.

이 단계에서는 아직 평가 수를 반영하지 않으므로, 뒤에서 Weighted Rating으로 보정한다.


In [43]:
def recommend_genre_similar_movies(title, topn=10):

    # 타이틀이 일치하는 영화의 인덱스 찾기
    index = movie_df[movie_df['title'] == title].index.values[0]

    # 장르가 유사한 영화 상위 topn * 2 조회
    rcmd_movie_df = movie_df.iloc[movie_idx_by_genre_sim_desc[index, :topn * 2]]

    # 추천 결과에 같은 영화(자기 자신)이 있지 못하도록 제외
    rcmd_movie_df = rcmd_movie_df[rcmd_movie_df.index != index]

    # 후보 중 편균 평점이 높은 영화 topn 개를 반환
    return rcmd_movie_df.sort_values(by='vote_average', ascending=False)[:topn]

#  함수 호출 테스트
recommend_genre_similar_movies('The Godfather', 15)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
1881,25000000,Drama Crime,NaN,278,prison corruption police brutality prison cell...,en,The Shawshank Redemption,Framed in the 1940s for the double murder of h...,136.747729,"[{""name"": ""Castle Rock Entertainment"", ""id"": 97}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",1994-09-23,28341469,142.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,Fear can hold you prisoner. Hope can set you f...,The Shawshank Redemption,8.5,8205
2731,13000000,Drama Crime,NaN,240,italo-american cuba vororte melancholy praise ...,en,The Godfather: Part II,In the continuing saga of the Corleone crime f...,105.792936,"[{""name"": ""Paramount Pictures"", ""id"": 4}, {""na...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",1974-12-20,47542841,200.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,"I don't feel I have to wipe everybody out, Tom...",The Godfather: Part II,8.3,3338
1847,25000000,Drama Crime,http://www.warnerbros.com/goodfellas,769,prison based on novel florida 1970s mass murde...,en,GoodFellas,"The true story of Henry Hill, a half-Irish, ha...",63.654244,"[{""name"": ""Winkler Films"", ""id"": 8880}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",1990-09-12,46836394,145.0,"[{""iso_639_1"": ""it"", ""name"": ""Italiano""}, {""is...",Released,Three Decades of Life in the Mafia.,GoodFellas,8.2,3128
1663,30000000,Drama Crime,NaN,311,life and death corruption street gang rape sad...,en,Once Upon a Time in America,A former Prohibition-era Jewish gangster retur...,49.336397,"[{""name"": ""Warner Bros."", ""id"": 6194}, {""name""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",1984-02-16,0,229.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,"Crime, passion and lust for power - Sergio Leo...",Once Upon a Time in America,8.2,1069
3454,6000000,Drama Crime Thriller,http://www.mgm.com/#/our-titles/2083/The-Usual...,629,law relatives theft criminal criminal mastermi...,en,The Usual Suspects,"Held in an L.A. interrogation room, Verbal Kin...",64.025031,"[{""name"": ""Blue Parrot Productions"", ""id"": 361...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",1995-07-19,23341568,106.0,"[{""iso_639_1"": ""es"", ""name"": ""Espa\u00f1ol""}, ...",Released,Five Criminals. One Line Up. No Coincidence.,The Usual Suspects,8.1,3254
3866,3300000,Drama Crime,http://cidadededeus.globo.com/,598,male nudity street gang brazilian photographer...,pt,Cidade de Deus,Cidade de Deus is a shantytown that started du...,44.356711,"[{""name"": ""O2 Filmes"", ""id"": 345}, {""name"": ""V...","[{""iso_3166_1"": ""BR"", ""name"": ""Brazil""}, {""iso...",2002-02-05,30641770,130.0,"[{""iso_639_1"": ""pt"", ""name"": ""Portugu\u00eas""}]",Released,"If you run you're dead... if you stay, you're ...",City of God,8.1,1814
3887,4000000,Drama Crime,http://www.miramax.com/movie/trainspotting/,627,london england alcohol sex based on novel drug...,en,Trainspotting,"Renton, deeply immersed in the Edinburgh drug ...",63.513324,"[{""name"": ""Miramax Films"", ""id"": 14}, {""name"":...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""}]",1996-02-23,16491080,93.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,Choose life.,Trainspotting,7.8,2655
892,52000000,Drama Crime,NaN,524,poker drug abuse 1970s overdose illegal prosti...,en,Casino,The life of the gambling paradise – Las Vegas ...,40.066880,"[{""name"": ""Universal Pictures"", ""id"": 33}, {""n...","[{""iso_3166_1"": ""FR"", ""name"": ""France""}, {""iso...",1995-11-22,116112375,178.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,No one stays at the top forever.,Casino,7.8,1307
883,52000000,Drama Crime,https://www.facebook.com/CatchMeIfYouCanMovie,640,con man biograph

### 평가 횟수에 대한 가중치가 부여된 평점(Weighted Rating) 적용

평균 평점만 보면 평가 수가 적은 영화가 상위에 올라오는 문제가 생길 수 있다.

예를 들어 1명이 평가해서 10점을 준 영화와 10,000명이 평가해서 평균 8.5점을 받은 영화가 있을 때, 단순 평균만 보면 첫 번째 영화가 더 좋아 보인다. 하지만 추천 결과에서는 평가 수가 충분한 영화가 더 신뢰도 높을 수 있다.

그래서 Weighted Rating은 평균 평점과 평가 수를 함께 반영한다.

$$
가중 평점(Weighted Rating) = (v/(v+m)) * R + (m/(v+m)) * C
$$

| 기호 | 의미 | 예시 |
| -- | -- | -- |
| v | 이 영화에 평점을 준 사람 수 `vote_count` | 예: 30명 |
| R | 이 영화의 평균 평점 `vote_average` | 예: 9.0점 |
| m | 믿을 만한 평가가 되려면 필요한 최소 평가 수 | 예: 100명 |
| C | 전체 영화의 평균 평점 | 예: 7.0점 |

평가 수가 많으면 해당 영화의 평균 평점이 더 많이 반영되고, 평가 수가 적으면 전체 평균 점수 쪽으로 보정된다.


## 17. 평가 수 분포 확인

Weighted Rating을 적용하기 전에 `vote_count` 분포를 확인한다.

평가 수가 매우 적은 영화는 평균 평점이 높더라도 신뢰도가 낮을 수 있으므로, 평가 수를 함께 고려해야 한다.


In [25]:
movie_df['vote_count'].describe()


count     4803.000000
mean       690.217989
std       1234.585891
min          0.000000
25%         54.000000
50%        235.000000
75%        737.000000
max      13752.000000
Name: vote_count, dtype: float64

## 18. Weighted Rating 기준값 계산

가중평점 계산에 필요한 최소 투표 수 기준 `m`과 전체 평균 평점 `C`를 계산한다.

`m`은 어느 정도 평가 수가 있어야 신뢰할 수 있는 영화로 볼지 정하는 기준이고, `C`는 평가 수가 적은 영화를 전체 평균 쪽으로 보정할 때 사용한다.


In [45]:
# m : 최소 투표 수 (평점을 등록한 사람의 수)
# quantile() : 분위수
# quantile(0.6) ==
m = movie_df[('vote_count')].quantile(0.6)
print('m:', m) # 하위 60% 지점의 투표수 :370.2
# 상위 40% (370.2표 이상 투표한) 영화

# C: 전체 영화의 평균 평점
C = movie_df['vote_average'].mean()
print('C:', C) # C: 6.09


m: 370.1999999999998
C: 6.092171559442016


## 19. 가중평점 컬럼 생성

평균 평점과 평가 수를 함께 반영한 `weighted_vote_average` 컬럼을 만든다.

이 값은 단순히 평점이 높은 영화보다, 충분한 평가 수를 가진 영화가 추천 상위에 오도록 보정하는 역할을 한다.


In [47]:
# 영화 한 행을 전달 받아 가중 평점을 계산해서 반환
def weighted_vote_average(movie):
    # m, C, v, R
    # v = 해당 영화 투표 수
    v = movie['vote_count']

    # R = 해당 영화의 평균 평점
    R = movie['vote_average']

    return (v / (v+m)) * R + (m / (v+m)) * C

# 가중 평점 계산
movie_df['weighted_vote_average'] = movie_df.apply(weighted_vote_average, axis=1)

movie_df.head()


,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,weighted_vote_average
0,237000000,Action Adventure Fantasy Science Fiction,http://www.avatarmovie.com/,19995,culture clash future space war space colony so...,en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...",...,2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,7.166301
1,300000000,Adventure Fantasy Action,http://disney.go.com/disneypictures/pirates/,285,ocean drug abuse exotic island east india trad...,en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...",...,2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,6.838594
2,245000000,Action Adventure Crime,http://www.sonypictures.com/movies/spectre/,206647,spy based on novel secret agent sequel mi6 bri...,en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...",...,2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466,6.284091
3,250000000,Action Crime Drama Thriller,http://www.thedarkknightrises.com/,49026,dc comics crime fighter terrorist secret ident...,en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...",...,2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106,7.541095
4,260000000,Action Adventure Science Fiction,http://movies.disney.com/john-carter,49529,based on novel mars medallion space travel pri...,en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]",...,2012-03-07,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124,6.098838


## 20. 단순 평균 평점과 가중평점 비교

단순 평균 평점 기준 상위 영화와 가중평점 값을 함께 비교한다.

평균 평점은 높지만 평가 수가 적은 영화가 어떻게 보정되는지 확인하는 단계이다.


In [49]:
movie_df[['title', 'vote_average', 'vote_count', 'weighted_vote_average']]\
    .sort_values('vote_average', ascending=False)[:10]

,title,vote_average,vote_count,weighted_vote_average
3519,Stiff Upper Lips,10.0,1,6.102699
4247,Me You and Five Bucks,10.0,2,6.113170
4045,"Dancer, Texas Pop. 81",10.0,1,6.102699
4662,Little Big Top,10.0,1,6.102699
3992,Sardaarji,9.5,2,6.110483
2386,One Man's Hero,9.3,2,6.109409
2970,There Goes My Baby,8.5,2,6.105110
1881,The Shawshank Redemption,8.5,8205,8.396052
2796,The Prisoner of Zenda,8.4,11,6.158767
3337,The Godfather,8.4,5893,8.263591


## 21. 최종 콘텐츠 기반 추천 함수

최종 추천 함수는 두 기준을 함께 사용한다.

- 1차 기준: 기준 영화와 장르가 비슷한 후보를 찾는다.
- 2차 기준: 후보 안에서 가중평점이 높은 영화를 위로 정렬한다.

즉, “비슷한 영화”와 “평가 신뢰도가 높은 영화”를 함께 고려하는 콘텐츠 기반 추천이다.


In [50]:
def recommend_genre_similar_movies(title, topn=10):
    try:
        # 입력된 title과 일치하는 제목의 영화를 찾아 인덱스 반환
        # 단, title이 일치하는 영화가 없으면 오류 발생!
        index = movie_df[movie_df['title'] == title].index.values[0]
    except:
        raise ValueError(f'해당 영화는 존재하지 않습니다: {title}')

    # 장르 유사도 기준 상위 영화 후보 조회
    rcmd_movie_df = movie_df.iloc[movie_idx_by_genre_sim_desc[index, :topn * 2]]

    # 추천 결고에서 자기 자신(같은 영화) 제외
    rcmd_movie_df = rcmd_movie_df[rcmd_movie_df.index != index]

     # 영화 추천 후보에 가중 평점을 계산
    return rcmd_movie_df.sort_values(by='vote_average', ascending=False)[:topn]

In [51]:
recommend_genre_similar_movies('Avatar', topn=15)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,weighted_vote_average
3208,0,Action Adventure Animation Fantasy Science Fic...,http://www.starwars.com/tv-shows/star-wars-the...,333355,war space opera clone troopers,en,Star Wars: Clone Wars (Volume 1),"The Saga continues with the Emmy-winning ""Star...",1.881466,"[{""name"": ""Lucasfilm"", ""id"": 1}, {""name"": ""Car...",...,2005-03-21,0,69.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,NaN,Star Wars: Clone Wars: Volume 1,8.0,27,6.221858
85,170000000,Action Adventure Science Fiction,http://www.captainamericathewintersoldiermovie...,100402,washington d.c. future shield marvel comic sup...,en,Captain America: The Winter Soldier,After the cataclysmic events in New York with ...,72.225265,"[{""name"": ""Marvel Studios"", ""id"": 420}]",...,2014-03-20,714766572,136.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,In heroes we trust.,Captain America: The Winter Soldier,7.6,5764,7.509002
46,250000000,Action Adventure Fantasy Science Fiction,http://www.x-menmovies.com/,127585,1970s mutant time travel marvel comic based on...,en,X-Men: Days of Future Past,The ultimate X-Men ensemble fights a war for t...,118.078691,"[{""name"": ""Twentieth Century Fox Film Corporat...",...,2014-05-15,747862775,131.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"To save the future, they must alter the past",X-Men: Days of Future Past,7.5,6032,7.418594
207,65000000,Action Adventure Science Fiction,NaN,861,oxygen falsely accused resistance mars double ...,en,Total Recall,Construction worker Douglas Quaid discovers a ...,43.129703,"[{""name"": ""TriStar Pictures"", ""id"": 559}, {""na...",...,1990-06-01,261317921,113.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"They stole his mind, now he wants it back.",Total Recall,7.1,1710,6.920643
182,130000000,Science Fiction Action Adventure,http://marvel.com/movies/movie/180/ant-man,102899,marvel comic superhero based on comic book aft...,en,Ant-Man,Armed with the astonishing ability to shrink i...,120.093610,"[{""name"": ""Marvel Studios"", ""id"": 420}]",...,2015-07-14,519311965,117.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,Heroes don't get any bigger.,Ant-Man,7.0,5880,6.946229
813,55000000,Action Adventure Fantasy Science Fiction,NaN,1924,saving the world journalist dc comics crime fi...,en,Superman,Mild-mannered Clark Kent works as a reporter a...,48.507081,"[{""name"": ""Warner Bros."", ""id"": 6194}, {""name""...",...,1978-12-13,300218018,143.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,You'll Believe a Man Can Fly!,Superman,6.9,1022,6.685190
56,185000000,Action Adventure Science Fiction,http://www.startrekmovie.com/,188927,sequel stranded hatred space opera,en,Star Trek Beyond,The USS Enterprise crew explores the furthest ...,65.352913,"[{""name"": ""Paramount Pictures"", ""id"": 4}, {""na...",...,2016-07-07,343471816,122.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,NaN,Star Trek Beyond,6.6,2568,6.536016
870,54000000,Action Adventure Fantasy Science Fiction,NaN,8536,saving the world dc comics sequel superhero ba...,en,Superman II,Three escaped criminals from the planet Krypto...,30.515175,"[{""name"": ""Warner Bros."", ""id"": 6194}, {""name""...",...,1980-12-04,190458706,127.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Man of Steel meets his match!,Superman II,6.5,629,6.348901
14,225000000,Action Adventure Fantasy Science Fiction,http://www.manofsteel.com/,49521,saving the world dc comics superhero based on ...,en,Man of Steel,A young boy learns that he has extraordinary p...,99.398009,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...",...,2013-06-12,662845518,143.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,You will believe that a man can fly.,M

## 22. 여러 기준 영화로 추천 결과 비교

마지막으로 기준 영화를 여러 개 바꿔가며 추천 결과가 어떻게 달라지는지 확인한다.

콘텐츠 기반 추천에서는 기준 영화의 장르 feature가 바뀌면 유사도 상위 후보도 달라진다.  
따라서 같은 추천 함수라도 입력 영화가 달라지면 추천 목록도 달라져야 한다.


In [53]:
sample_titles = ['Avatar', 'The Godfather', 'Toy Story']

recommendation_results = []

for base_title in sample_titles:
    # recommend_genre_similar_movies(): 기준 영화와 장르가 비슷하고 가중평점이 높은 영화를 반환
    result_df = recommend_genre_similar_movies(base_title, topn=5).copy()

    # 어떤 기준 영화에서 나온 추천인지 구분하기 위해 컬럼을 추가
    result_df.insert(0, 'base_title', base_title)

    recommendation_results.append(
        result_df[['base_title', 'title', 'genres', 'vote_average', 'vote_count', 'weighted_vote_average']]
    )

multi_recommendation_df = pd.concat(recommendation_results, ignore_index=True)
display(multi_recommendation_df)

,base_title,title,genres,vote_average,vote_count,weighted_vote_average
0,Avatar,X-Men: Days of Future Past,Action Adventure Fantasy Science Fiction,7.5,6032,7.418594
1,Avatar,Superman,Action Adventure Fantasy Science Fiction,6.9,1022,6.685190
2,Avatar,Superman II,Action Adventure Fantasy Science Fiction,6.5,629,6.348901
3,Avatar,Man of Steel,Action Adventure Fantasy Science Fiction,6.5,6359,6.477564
4,Avatar,Hellboy II: The Golden Army,Adventure Fantasy Science Fiction,6.5,1527,6.420421
5,The Godfather,GoodFellas,Drama Crime,8.2,3128,7.976937
6,The Godfather,City of God,Drama Crime,8.1,1814,7.759693
7,The Godfather,This Is England,Drama Crime,7.4,363,6.739664
8,The Godfather,American Gangster,Drama Crime,7.4,1502,7.141396
9,The Godfather,Rounders,Drama Crime,6.9,439,6.530427
